In [1]:
import os
import re
import json
from tqdm import tqdm
from pathlib import Path
from pydantic import BaseModel, Field
from typing import List
from utils import get_llm, get_response_text, get_concepts, google_api_key_list, gemini_model_list

In [2]:
description_prompt_template = """\
Тебе будет дано содержание (CONTENT) статьи, ты должен очень-очень кратко описать, о чём статья (максимум одно предложение).
Пример: "О фундаментальном понятии материи..."

CONTENT:
{content}
"""

lemmas_prompt_template = """\
Тебе будет дано содержание (CONTENT) статьи, ты должен перечислить основные ключевые слова - понятия, которые можно узнать из этой статьи.
Важно: твой ответ должен быть корректным списком, слова должны перечисляться через запятую. Ничего кроме перечисленных слов не пиши.

CONTENT:
{content}
"""

In [3]:
llm_description = get_llm('Gigachat')
llm_lemmas = get_llm('Gigachat')

In [4]:
concepts = get_concepts()
concepts[:5]

[('Q1', 'Вселенная'),
 ('Q102836', 'пружина'),
 ('Q104837', 'термодинамическая фаза'),
 ('Q1075', 'цвет'),
 ('Q107715', 'физическая величина')]

In [5]:
filename = "concepts.json"

if os.path.exists(filename):
    with open(filename, "r", encoding="utf-8") as f:
        information = json.load(f)
else:
    information = {
        "section": "1.2_natural_sciences",
        "topic": "physics_in_everyday_life",
        "author_group": "Команда 4",
        "members": [
          "Ивченко Матвей",
          "Клюкин Павел",
          "Найпак Дмитрий",
          "Пермяков Герман"
        ],
        "concepts": []
    }

In [6]:
papers_dir = f"WEB/{information['section']}/{information['topic']}"

In [7]:
def read_concept(concept_id: str):
    with open(Path.cwd().parents[2] / papers_dir / f'{concept_id}.md', mode='r', encoding='utf-8') as file:
        return '\n'.join(file.readlines())

In [8]:
def get_titles_from_text(text: str):
    # Ищем заголовок "## Содержание" напрямую
    match = re.search(r'^##\s+Содержание\s*$', text, re.MULTILINE)
    if match is None:
        raise ValueError('Блок "## Содержание" не найден в тексте')
    
    # Берём всё от "## Содержание" до следующего заголовка уровня ## или конца
    index_block = text[match.start():]
    end = re.search(r'^##\s+', index_block[3:], re.MULTILINE)  # ищем следующий ##
    if end:
        index_block = index_block[:end.start() + 3]

    index_block = '\n'.join(index_block.split('\n')[1:])
    index_block = index_block
    titles = re.findall(r'\[\d+(?:\.\d+)*\.\s*(.*?)\]', index_block)
    titles = [f'- {title}' for title in titles]
    return '\n'.join(titles)

In [9]:
def is_concept_in_info(concept_id):
    for concept_info in information['concepts']:
        if str(concept_id) == str(concept_info['wikidata_id']):
            return True
        
    return False

In [10]:
for concept_id, concept_name in tqdm(concepts):
    if is_concept_in_info(concept_id):
        continue

    concept_info = {
        "id": papers_dir,
        "file": f"{papers_dir}/{concept_id}.md",
        "wikidata_id": f"{concept_id}",
        "author": "gemini-3.1-flash-lite-preview"
    }

    article = read_concept(str(concept_id))
    index = get_titles_from_text(article)

    desc_prompt = description_prompt_template.format(content=index)
    concept_info["description"] = get_response_text(llm_description.invoke(desc_prompt))

    lem_prompt = lemmas_prompt_template.format(content=index)
    lemmas = get_response_text(llm_lemmas.invoke(lem_prompt)).split(',')
    lemmas = [word.strip() for word in lemmas]
    concept_info["lemmas"] = lemmas

    information["concepts"].append(concept_info)

    # Сохранение после каждого успешного вызова LLM
    with open("concepts.json", "w", encoding="utf-8") as f:
        json.dump(information, f, ensure_ascii=False, indent=4)

  0%|          | 0/168 [00:00<?, ?it/s]

100%|██████████| 168/168 [00:06<00:00, 24.93it/s]
